# BioDiscML Report

Interactive exploration of a BioDiscML run: model leaderboard, and for each
model selected by `-bestmodel` -- a PCA, a hierarchically-clustered heatmap,
a confusion matrix, and per-feature boxplots of its biomarker signature
(hover for details, zoom, click the legend to filter).

Charts are interactive [Plotly](https://plotly.com/python/) figures, but the
exported HTML is fully self-contained -- nothing to host.

**Expected input**: a BioDiscML run's output folder (`-train` then
`-bestmodel`), containing at least:
- `<project>_c.*results.csv` (the leaderboard)
- `<project>_d.<model>....train_features.csv` or `...all_features.csv` (one
  per model selected by `-bestmodel`; the suffix depends on whether a
  separate test set was used, per BioDiscML's own `readme.md`)


In [ ]:
import glob
import json
import os
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster.hierarchy import dendrogram as scipy_dendrogram
from scipy.cluster.hierarchy import linkage
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]

INFO_BOX_STYLE = (
    "margin:0.5em 0 1em 0; padding:0.8em 1.1em; border:1px solid #d0d0d8; "
    "border-radius:6px; background:#f7f7fa;"
)

_plotlyjs_included = False


def show(fig):
    # nbconvert (headless kernel execution) doesn't reliably render
    # Plotly's native application/vnd.plotly.v1+json mimebundle into
    # static HTML across templates/versions -- emitting an explicit
    # text/html output via to_html() always embeds correctly instead.
    # include_plotlyjs=True (not "cdn") embeds the whole library inline,
    # so the exported HTML report is fully self-contained: no network,
    # no external CDN, works offline, forever.
    global _plotlyjs_included
    include_js = True if not _plotlyjs_included else False
    display(HTML(fig.to_html(full_html=False, include_plotlyjs=include_js)))
    _plotlyjs_included = True

## Parameters

Read from the `BIODISCML_OUTPUT_DIR` / `PROJECT` / `SORT_METRIC` environment variables if set (handy for `generate_report.sh` or the command line), otherwise edit the defaults below.

In [ ]:
# Folder containing the _c./_d. files produced by
# `java -jar $BIODISCML_JAR -config ... -train -bestmodel`
BIODISCML_OUTPUT_DIR = os.environ.get("BIODISCML_OUTPUT_DIR", ".")

# Project name, as set by `project=` in BioDiscML's config.conf
PROJECT = os.environ.get("PROJECT", "contrastA")

# Metric used to sort the model table -- AVG_MCC is the real default of
# BioDiscML's own config.conf for -bestmodel (numberOfBestModelsSortingMetric
# =AVG_MCC), so it's also the default here: the retained model(s) then show
# up near the top of the leaderboard in most runs, instead of TRAIN_10CV_MCC
# which saturates fast (many models tied at a perfect score) on small
# sample sizes.
SORT_METRIC = os.environ.get("SORT_METRIC", "AVG_MCC")


## Model leaderboard (`_c.*results.csv`)

In [ ]:
results_files = glob.glob(os.path.join(BIODISCML_OUTPUT_DIR, f"{PROJECT}_c.*results.csv"))
if not results_files:
    raise FileNotFoundError(
        f"No '{PROJECT}_c.*results.csv' file found in {BIODISCML_OUTPUT_DIR!r}. "
        "Check BIODISCML_OUTPUT_DIR and PROJECT."
    )

results = pd.read_csv(results_files[0], sep="\t")

# BioDiscML writes undefined metrics (e.g. an MCC on a fold with a single
# class, common on small datasets) as NaN/Infinity (1.9), or with Java
# DecimalFormat's symbols (U+FFFD, "∞") in older versions; the .632+ column
# is empty when bootstrap is disabled. Left as-is, pandas can read a column as text:
# sorting would then be alphabetical instead of numeric. Coerce every metric
# column to numbers; anything undefined (non-numeric or infinite) becomes NaN,
# sorted last, rather than letting an "infinite" score rank first.
TEXT_COLUMNS = {"ID", "classifier", "Options", "OptimizedValue", "SearchMode"}
for col in results.columns:
    if col in TEXT_COLUMNS or col.endswith("matrix"):
        continue
    results[col] = pd.to_numeric(results[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

# Same rules as BioDiscML's -bestmodel: the metric name is not
# case-sensitive, and error rates are sorted in ascending order.
sort_metric = next((c for c in results.columns if c.upper() == SORT_METRIC.upper()), results.columns[6])
SORT_ASCENDING = any(
    k in sort_metric.upper()
    for k in ("RMSE", "BER", "FPR", "FNR", "FDR", "MAE", "RAE", "RRSE", ".632")
)
results_sorted = results.sort_values(sort_metric, ascending=SORT_ASCENDING).reset_index(drop=True)

# Reconstructs, for each row, the model name as it appears in the
# _d.*features.csv filenames (classifier_options_optimized-metric_search-mode)
# -- lets us find the exact rank of a model retained by -bestmodel, even
# when that rank is outside the "top 10" shown below (common: several
# models can be tied on the sorting metric, especially with few samples).
# BioDiscML strips spaces, but also quotes and backslashes, from the options
# in filenames: nested Weka options (IBk, LWL, SMO...) look like
# `-A "weka...LinearNNSearch -A \"weka...EuclideanDistance -R first-last\""`.
results_sorted["_core"] = (
    results_sorted["classifier"].astype(str) + "_"
    + results_sorted["Options"].fillna("").astype(str).str.replace(r'[\s"\\]', "", regex=True) + "_"
    + results_sorted["OptimizedValue"].astype(str) + "_"
    + results_sorted["SearchMode"].astype(str)
)

top = results_sorted.head(15)[["ID", "classifier", "SearchMode", "nbrOfFeatures", sort_metric]]
top

In [ ]:
# Model(s) actually extracted by -bestmodel in this folder (the
# _d.*features.csv files) -- computed here so they can be highlighted in the
# chart below, even when they fall outside the natural "top 10" of the
# sorting metric (common with few samples: many models end up perfectly
# tied on a given metric).
feature_files = sorted(
    f for f in glob.glob(os.path.join(BIODISCML_OUTPUT_DIR, f"{PROJECT}_d.*features.csv"))
    if "corrfeatures" not in os.path.basename(f).lower()
)
retained_cores = {
    os.path.basename(f).split(f"{PROJECT}_d.")[-1].rsplit(".", 2)[0] for f in feature_files
}

top10 = results_sorted.head(10)
retained_rows = results_sorted[results_sorted["_core"].isin(retained_cores)]
plot_data = (
    pd.concat([top10, retained_rows])
    .drop_duplicates(subset="ID")
    .sort_values(sort_metric, ascending=SORT_ASCENDING)
    .copy()
)
plot_data["Status"] = plot_data["_core"].apply(
    lambda c: "Model(s) retained by -bestmodel" if c in retained_cores else "Top 10"
)

# How saturated is this metric? On small sample sizes it's common for a
# large fraction of models to be tied at the best achievable value -- in
# which case a model's exact rank (204th, 52nd...) is not meaningful on its
# own. Surfaced here directly, for whichever metric is actually displayed,
# rather than left as something only discoverable by investigating.
best_value = results_sorted[sort_metric].min() if SORT_ASCENDING else results_sorted[sort_metric].max()
n_tied = int((results_sorted[sort_metric] == best_value).sum())
n_total = len(results_sorted)
tie_note = ""
if n_tied > 1:
    tie_pct = 100 * n_tied / n_total
    tie_note = (
        f"<br><sub>{n_tied}/{n_total} models ({tie_pct:.0f}%) are tied at "
        f"the best {sort_metric} value ({best_value:.3g}) -- common on small "
        "sample sizes; a model's exact rank within that tied group isn't "
        "meaningful on its own.</sub>"
    )

fig = px.bar(
    plot_data,
    x=sort_metric,
    y="ID",
    color="Status",
    orientation="h",
    hover_data=["classifier", "SearchMode", "nbrOfFeatures"],
    color_discrete_map={
        "Model(s) retained by -bestmodel": "#C44E52",
        "Top 10": "#4C72B0",
    },
    title=f"Top 10 models by {sort_metric}, with the model(s) retained by -bestmodel highlighted{tie_note}",
)
fig.update_layout(
    yaxis={"categoryorder": "total descending" if SORT_ASCENDING else "total ascending"},
    height=max(450, 30 * len(plot_data)),
    margin=dict(t=80 if tie_note else 60),
)
show(fig)

## Retained models summary

Key statistics for the models actually extracted by `-bestmodel` (the same rows highlighted in red above), side by side.

In [ ]:
summary_cols = ["ID", "classifier", "nbrOfFeatures"]
for m in [sort_metric, "AVG_MCC", "TRAIN_TEST_RH_MCC",
          "TRAIN_10CV_ACC", "TRAIN_10CV_AUC", "TRAIN_10CV_MCC", "TRAIN_10CV_BER"]:
    if m in results_sorted.columns and m not in summary_cols:
        summary_cols.append(m)

summary_table = (
    results_sorted[results_sorted["_core"].isin(retained_cores)][summary_cols]
    .sort_values(sort_metric, ascending=SORT_ASCENDING)
    .reset_index(drop=True)
)
summary_table

## Consensus signature across retained models

A feature that recurs across several independently retained models is less likely to be an artifact of one particular classifier/feature-subset search, and more likely to be robust signal. This compares the feature lists of every model retained by `-bestmodel` above.

In [ ]:
def class_column_name(df):
    # BioDiscML always names the last column 'class'.
    return df.columns[-1]


if len(feature_files) < 2:
    print("Only one retained model here -- nothing to compare a consensus against.")
else:
    model_features = {}
    for f in feature_files:
        name = os.path.basename(f).split(f"{PROJECT}_d.")[-1].rsplit(".", 2)[0]
        df_f = pd.read_csv(f, index_col=0)
        class_col_f = class_column_name(df_f)
        model_features[name] = set(c for c in df_f.columns if c != class_col_f)

    n_models = len(model_features)
    all_features = sorted(set.union(*model_features.values()))
    freq = pd.DataFrame({
        "feature": all_features,
        "n_models": [sum(feature in feats for feats in model_features.values()) for feature in all_features],
    })
    freq["models"] = freq["feature"].apply(
        lambda feature: ", ".join(name for name, feats in model_features.items() if feature in feats)
    )
    freq = freq.sort_values(["n_models", "feature"], ascending=[False, True]).reset_index(drop=True)

    consensus_features = freq.loc[freq["n_models"] == n_models, "feature"].tolist()
    if consensus_features:
        consensus_note = (
            f"<strong>{len(consensus_features)} feature(s)</strong> appear in "
            f"all {n_models} retained models: {', '.join(consensus_features)}."
        )
    else:
        consensus_note = (
            f"No single feature appears in all {n_models} retained models -- "
            "the retained signatures don't fully overlap (still common: each "
            "model can reach a similar score via a different feature subset)."
        )
    display(HTML(f'<div style="{INFO_BOX_STYLE}">{consensus_note}</div>'))

    fig_consensus = px.bar(
        freq, x="n_models", y="feature", orientation="h",
        color=(freq["n_models"] == n_models).map({True: "In all retained models", False: "In some"}),
        color_discrete_map={"In all retained models": "#C44E52", "In some": "#4C72B0"},
        hover_data=["models"],
        labels={"n_models": "Number of retained models containing this feature", "feature": "", "color": ""},
        title=f"Feature recurrence across the {n_models} retained models",
    )
    fig_consensus.update_layout(
        legend_title_text="",
        yaxis={"categoryorder": "total ascending"},
        height=max(350, 24 * len(freq)),
        xaxis=dict(dtick=1, range=[0, n_models + 0.2]),
    )
    show(fig_consensus)

## Best model signatures (`_d.*features.csv`)

A dropdown lets you pick which model to display -- normally one per model retained by `-bestmodel` (`numberOfBestModels` in the config.conf), already highlighted **in red** in the chart above.

In [ ]:
def class_column_name(df):
    # BioDiscML always names the last column 'class'.
    return df.columns[-1]


def find_rank(model_name):
    match = results_sorted.index[results_sorted["_core"] == model_name]
    if len(match) == 0:
        return None
    idx = match[0]
    metric_value = results_sorted.loc[idx, sort_metric]
    n_tied = int((results_sorted[sort_metric] == metric_value).sum())
    return idx + 1, len(results_sorted), metric_value, n_tied


def find_confusion_matrix(model_name):
    # BioDiscML's "TRAIN_matrix" column holds the 10-fold CV confusion
    # matrix on the training split, as a compact "[a b] [c d]" string --
    # without stating which row/column is which class.
    match = results_sorted.index[results_sorted["_core"] == model_name]
    if len(match) == 0 or "TRAIN_matrix" not in results_sorted.columns:
        return None
    raw = results_sorted.loc[match[0], "TRAIN_matrix"]
    if not isinstance(raw, str):
        return None
    pairs = re.findall(r"\[(-?\d+)\s+(-?\d+)\]", raw)
    if len(pairs) != 2:
        return None
    return [[int(pairs[0][0]), int(pairs[0][1])], [int(pairs[1][0]), int(pairs[1][1])]]


def confusion_matrix_html(model_name, class_counts_full):
    # Matches the matrix's row totals against the known full-sample class
    # counts (each row must be a subset of one class) to recover which row
    # is which class -- BioDiscML's compact matrix format doesn't say. If
    # that's ambiguous (rare), rows are labelled generically rather than
    # risking a wrong label.
    matrix = find_confusion_matrix(model_name)
    if matrix is None:
        return ""
    classes = sorted(class_counts_full)
    if len(classes) != 2:
        return ""
    a, b = classes
    row_totals = [sum(matrix[0]), sum(matrix[1])]
    candidates = []
    if row_totals[0] <= class_counts_full[a] and row_totals[1] <= class_counts_full[b]:
        candidates.append((a, b))
    if row_totals[0] <= class_counts_full[b] and row_totals[1] <= class_counts_full[a]:
        candidates.append((b, a))
    if len(candidates) == 1:
        row_a, row_b = candidates[0]
        note = ""
    else:
        row_a, row_b = "Class 1", "Class 2"
        note = " (BioDiscML doesn't label rows/columns, and the exact class order couldn't be confirmed from the data available here)"
    n = sum(row_totals)
    return f"""
<div style="margin:0.8em 0;">
<strong>Confusion matrix</strong> — 10-fold cross-validation on the training
split (n={n}, may differ from the sample count above, which covers the full
signature file){note}.
<table style="border-collapse:collapse; margin-top:0.4em; font-size:0.95em;">
<tr>
  <td></td><td></td>
  <td colspan="2" style="text-align:center; padding:0.3em 0.6em; border-bottom:1px solid #ccc;"><em>Predicted</em></td>
</tr>
<tr>
  <td></td><td></td>
  <td style="padding:0.3em 0.6em; text-align:center; font-weight:bold;">{row_a}</td>
  <td style="padding:0.3em 0.6em; text-align:center; font-weight:bold;">{row_b}</td>
</tr>
<tr>
  <td rowspan="2" style="writing-mode:vertical-rl; text-align:center; padding:0.3em; border-right:1px solid #ccc;"><em>Actual</em></td>
  <td style="padding:0.3em 0.6em; font-weight:bold;">{row_a}</td>
  <td style="padding:0.3em 0.6em; text-align:center; background:#eef3fa;">{matrix[0][0]}</td>
  <td style="padding:0.3em 0.6em; text-align:center;">{matrix[0][1]}</td>
</tr>
<tr>
  <td style="padding:0.3em 0.6em; font-weight:bold;">{row_b}</td>
  <td style="padding:0.3em 0.6em; text-align:center;">{matrix[1][0]}</td>
  <td style="padding:0.3em 0.6em; text-align:center; background:#eef3fa;">{matrix[1][1]}</td>
</tr>
</table>
</div>
"""


def model_stats(feature_csv, df, class_col, feature_cols, model_name):
    class_counts = ", ".join(f"{k} ({v})" for k, v in df[class_col].value_counts().items())

    rank_info = find_rank(model_name)
    if rank_info:
        rank, total, metric_value, n_tied = rank_info
        rank_html = (
            f"<strong>Rank {rank}/{total}</strong> by {sort_metric} "
            f"(value: {metric_value:.3g}) — highlighted in red in the chart "
            "above."
        )
        if n_tied > 1:
            rank_html += (
                f" Tied with {n_tied - 1} other model(s) at this exact "
                f"value ({n_tied}/{total} total) -- the specific rank "
                "number isn't meaningful within that tied group."
            )
    else:
        rank_html = (
            "Model not found in the leaderboard above "
            "(classifier/options match was inconclusive)."
        )

    return {
        "name": model_name,
        "n_features": len(feature_cols),
        "features": ", ".join(feature_cols),
        "n_samples": len(df),
        "class_counts": class_counts,
        "rank_html": rank_html,
    }


def stats_to_html(s):
    return (
        f"<h3 style=\"margin-top:0\">{s['name']}</h3>"
        f"<p><strong>{s['n_features']} feature(s)</strong> in the signature: "
        f"{s['features']}<br>"
        f"<strong>{s['n_samples']} sample(s)</strong>: {s['class_counts']}<br>"
        f"{s['rank_html']}</p>"
    )


def _dendrogram_traces(matrix, orientation):
    # matrix: observations (leaves) x variables. Returns Plotly line traces for
    # the dendrogram and the leaf order (indices into the original matrix).
    n = matrix.shape[0]
    if n < 2:
        return [], list(range(n))
    Z = linkage(matrix, method="average", metric="euclidean")
    d = scipy_dendrogram(Z, no_plot=True)
    leaves = d["leaves"]
    traces = []
    for xs, ys in zip(d["icoord"], d["dcoord"]):
        # scipy places leaf i at x = 10*i + 5 -- rescale to 0-indexed positions
        # so dendrogram leaves line up with the heatmap's integer tick positions.
        pos = [(v - 5) / 10 for v in xs]
        if orientation == "top":
            traces.append(go.Scatter(x=pos, y=ys, mode="lines",
                                      line=dict(color="#888", width=1),
                                      hoverinfo="skip", showlegend=False))
        else:
            traces.append(go.Scatter(x=ys, y=pos, mode="lines",
                                      line=dict(color="#888", width=1),
                                      hoverinfo="skip", showlegend=False))
    return traces, leaves


def clustered_heatmap(scaled_df, class_series, model_name):
    # scaled_df: samples (rows) x features (cols), z-scored.
    # class_series: class label per sample, aligned to scaled_df.index.
    n_samples, n_features = scaled_df.shape
    col_traces, col_leaves = _dendrogram_traces(scaled_df.values, "top")      # cluster samples
    row_traces, row_leaves = _dendrogram_traces(scaled_df.values.T, "left")   # cluster features

    sample_order = [scaled_df.index[i] for i in col_leaves]
    feature_order = [scaled_df.columns[i] for i in row_leaves]
    z = scaled_df.loc[sample_order, feature_order].values.T  # features x samples, reordered

    classes = sorted({str(c) for c in class_series.values})
    color_for = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(classes)}
    class_codes = [classes.index(str(class_series.loc[s])) for s in sample_order]
    n_classes = max(len(classes), 2)
    strip_colorscale = []
    for i, c in enumerate(classes):
        strip_colorscale.append([i / n_classes, color_for[c]])
        strip_colorscale.append([(i + 1) / n_classes, color_for[c]])

    fig = make_subplots(
        rows=3, cols=2,
        specs=[[None, {}], [{"rowspan": 2}, {}], [None, {}]],
        shared_xaxes=True,
        row_heights=[0.18, 0.06, 0.76] if col_traces else [0.001, 0.06, 0.939],
        column_widths=[0.16, 0.84] if row_traces else [0.001, 0.999],
        horizontal_spacing=0.01, vertical_spacing=0.015,
    )
    for tr in col_traces:
        fig.add_trace(tr, row=1, col=2)
    for tr in row_traces:
        fig.add_trace(tr, row=2, col=1)
    fig.add_trace(
        go.Heatmap(
            z=[class_codes], x=list(range(n_samples)),
            customdata=[[str(class_series.loc[s]) for s in sample_order]],
            colorscale=strip_colorscale, zmin=0, zmax=n_classes - 1, showscale=False,
            hovertemplate="Sample %{x}<br>Class: %{customdata}<extra></extra>",
        ),
        row=2, col=2,
    )
    fig.add_trace(
        go.Heatmap(
            z=z, x=list(range(n_samples)), y=list(range(n_features)),
            colorscale="RdBu_r", zmid=0, colorbar=dict(title="z-score", len=0.72, y=0.35),
        ),
        row=3, col=2,
    )

    fig.update_xaxes(row=3, col=2, tickmode="array", tickvals=list(range(n_samples)),
                      ticktext=sample_order, tickangle=45, side="bottom",
                      range=[-0.5, n_samples - 0.5])
    fig.update_yaxes(row=3, col=2, tickmode="array", tickvals=list(range(n_features)),
                      ticktext=feature_order, side="right", range=[-0.5, n_features - 0.5])
    fig.update_xaxes(row=1, col=2, showticklabels=False, showgrid=False, zeroline=False,
                      range=[-0.5, n_samples - 0.5])
    fig.update_yaxes(row=1, col=2, showticklabels=False, showgrid=False, zeroline=False)
    fig.update_xaxes(row=2, col=2, showticklabels=False, showgrid=False, zeroline=False,
                      range=[-0.5, n_samples - 0.5])
    fig.update_yaxes(row=2, col=2, showticklabels=False, showgrid=False, zeroline=False)
    fig.update_xaxes(row=2, col=1, showticklabels=False, showgrid=False, zeroline=False,
                      autorange="reversed")
    fig.update_yaxes(row=2, col=1, showticklabels=False, showgrid=False, zeroline=False,
                      range=[-0.5, n_features - 0.5])
    fig.update_xaxes(row=1, col=1, visible=False)
    fig.update_yaxes(row=1, col=1, visible=False)

    legend_note = " · ".join(f'<span style="color:{color_for[c]}">■</span> {c}' for c in classes)
    fig.update_layout(
        title=f"Hierarchically-clustered heatmap — {model_name}<br>"
              f"<sub>Color strip = true class: {legend_note}</sub>",
        height=max(440, 26 * n_features + 190),
        showlegend=False,
        margin=dict(t=80, b=80, l=10, r=80),
    )
    return fig


def _confidence_ellipse(x, y, confidence=0.95, n_points=100):
    # Ellipse based on the covariance of (x, y), scaled so it encloses the
    # given confidence level under a bivariate-normal assumption (chi-square
    # with 2 degrees of freedom). Returns None if there aren't enough points
    # (or they're perfectly collinear) to estimate a non-degenerate ellipse.
    if len(x) < 3:
        return None
    cov = np.cov(x, y)
    if np.any(np.isnan(cov)) or np.linalg.det(cov) <= 1e-12:
        return None
    mean_x, mean_y = np.mean(x), np.mean(y)
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    order = eigenvals.argsort()[::-1]
    eigenvals, eigenvecs = eigenvals[order], eigenvecs[:, order]
    radius = np.sqrt(stats.chi2.ppf(confidence, df=2))
    theta = np.linspace(0, 2 * np.pi, n_points)
    axes = radius * np.sqrt(eigenvals)
    circle = np.vstack([axes[0] * np.cos(theta), axes[1] * np.sin(theta)])
    ellipse = eigenvecs @ circle
    return ellipse[0] + mean_x, ellipse[1] + mean_y


def _hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def render_model_block(feature_csv):
    df = pd.read_csv(feature_csv, index_col=0)
    class_col = class_column_name(df)
    feature_cols = [c for c in df.columns if c != class_col]
    model_name = os.path.basename(feature_csv).split(f"{PROJECT}_d.")[-1].rsplit(".", 2)[0]

    model_info = model_stats(feature_csv, df, class_col, feature_cols, model_name)
    parts = []

    # --- PCA, with 95% confidence ellipse per class ---
    if len(feature_cols) >= 2:
        X = StandardScaler().fit_transform(df[feature_cols].values)
        pca_model = PCA(n_components=2)
        pcs = pca_model.fit_transform(X)
        var_pc1, var_pc2 = pca_model.explained_variance_ratio_[:2] * 100
        pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"], index=df.index)
        pca_df[class_col] = df[class_col].astype(str).values
        pca_df["sample"] = df.index

        pca_classes = sorted(pca_df[class_col].unique())
        color_for_pca = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(pca_classes)}

        fig_pca = go.Figure()
        any_ellipse = False
        for c in pca_classes:
            sub = pca_df[pca_df[class_col] == c]
            ellipse = _confidence_ellipse(sub["PC1"].values, sub["PC2"].values)
            if ellipse is not None:
                any_ellipse = True
                ex, ey = ellipse
                fig_pca.add_trace(go.Scatter(
                    x=ex, y=ey, mode="lines", fill="toself",
                    line=dict(color=color_for_pca[c], width=1),
                    fillcolor=_hex_to_rgba(color_for_pca[c], 0.15),
                    showlegend=False, hoverinfo="skip",
                ))
            fig_pca.add_trace(go.Scatter(
                x=sub["PC1"], y=sub["PC2"], mode="markers", name=c,
                marker=dict(size=11, color=color_for_pca[c]),
                text=sub["sample"],
                hovertemplate="%{text}<br>PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra>" + c + "</extra>",
            ))
        subtitle = " (ellipse = 95% confidence region)" if any_ellipse else ""
        fig_pca.update_layout(
            title=f"PCA — {model_name}{subtitle}",
            xaxis_title=f"PC1 ({var_pc1:.1f}% variance explained)",
            yaxis_title=f"PC2 ({var_pc2:.1f}% variance explained)",
        )
        parts.append(fig_pca.to_html(full_html=False, include_plotlyjs=False))

    # --- Hierarchically-clustered heatmap (samples and features) ---
    scaled = pd.DataFrame(
        StandardScaler().fit_transform(df[feature_cols]),
        index=df.index,
        columns=feature_cols,
    )
    fig_heat = clustered_heatmap(scaled, df[class_col], model_name)
    parts.append(fig_heat.to_html(full_html=False, include_plotlyjs=False))

    # --- Confusion matrix (10-fold CV on the training split) ---
    cm_html = confusion_matrix_html(model_name, df[class_col].value_counts().to_dict())
    if cm_html:
        parts.append(cm_html)

    # --- Interactive per-feature boxplots, with Mann-Whitney p-value for 2 classes ---
    classes = df[class_col].unique()
    n_cols = min(len(feature_cols), 4)
    n_rows = -(-len(feature_cols) // n_cols)
    titles = []
    for feature in feature_cols:
        if len(classes) == 2:
            g1 = df.loc[df[class_col] == classes[0], feature]
            g2 = df.loc[df[class_col] == classes[1], feature]
            _, p = stats.mannwhitneyu(g1, g2, alternative="two-sided")
            titles.append(f"{feature} (p={p:.3g})")
        else:
            titles.append(feature)

    fig_box = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titles)
    for i, feature in enumerate(feature_cols):
        row_i, col_i = i // n_cols + 1, i % n_cols + 1
        for cls in classes:
            fig_box.add_trace(
                go.Box(
                    y=df.loc[df[class_col] == cls, feature],
                    name=str(cls),
                    boxpoints="all",
                    jitter=0.4,
                    marker_color="#4C72B0" if cls == classes[0] else "#DD8452",
                    showlegend=(i == 0),
                    legendgroup=str(cls),
                ),
                row=row_i, col=col_i,
            )
    fig_box.update_layout(height=300 * n_rows, title=f"Per-feature distributions — {model_name}")
    parts.append(fig_box.to_html(full_html=False, include_plotlyjs=False))

    return model_info, "\n".join(parts)


if not feature_files:
    print(
        f"No '{PROJECT}_d.*features.csv' file found. "
        "Was this run followed by a `-bestmodel` step?"
    )
elif len(feature_files) == 1:
    # A single retained model: no need for a selector, static info panel.
    model_info, charts_html = render_model_block(feature_files[0])
    display(HTML(f'<div style="{INFO_BOX_STYLE}">{stats_to_html(model_info)}</div>{charts_html}'))
else:
    stats_by_index = []
    options_html = []
    blocks_html = []
    for i, f in enumerate(feature_files):
        model_info, charts_html = render_model_block(f)
        stats_by_index.append(model_info)
        options_html.append(f'<option value="{i}">{model_info["name"]}</option>')
        display_style = "block" if i == 0 else "none"
        blocks_html.append(
            f'<div class="biodiscml-model-block" id="biodiscml-model-{i}" '
            f'style="display:{display_style}">{charts_html}</div>'
        )

    # JSON.stringify-safe: escapes "</" so it can't prematurely close the
    # <script> tag if a feature/model name happened to contain that sequence.
    stats_json = json.dumps(stats_by_index).replace("</", "<\\/")

    selector_html = f"""
<div style="margin:0.5em 0 0.3em 0;">
  <label for="biodiscml-model-selector"><strong>Model to display:</strong></label>
  <select id="biodiscml-model-selector" style="margin-left:0.5em; padding:0.3em; font-size:1em;">
    {''.join(options_html)}
  </select>
</div>
<div style="font-size:0.85em; color:#666; margin-bottom:0.8em;">
  Only models actually extracted by <code>-bestmodel</code> in this folder
  are listed here (typically <code>numberOfBestModels</code> in the
  config.conf) — not every model tied in the leaderboard above.
</div>
<div id="biodiscml-model-info" style="{INFO_BOX_STYLE}"></div>
{''.join(blocks_html)}
<script>
(function() {{
    var stats = {stats_json};

    function statsToHtml(s) {{
        return '<h3 style="margin-top:0">' + s.name + '</h3>' +
            '<p><strong>' + s.n_features + ' feature(s)</strong> in the signature: ' +
            s.features + '<br>' +
            '<strong>' + s.n_samples + ' sample(s)</strong>: ' + s.class_counts + '<br>' +
            s.rank_html + '</p>';
    }}

    function selectModel(i) {{
        document.getElementById('biodiscml-model-info').innerHTML = statsToHtml(stats[i]);
        document.querySelectorAll('.biodiscml-model-block').forEach(function(el) {{ el.style.display = 'none'; }});
        var block = document.getElementById('biodiscml-model-' + i);
        block.style.display = 'block';
        block.querySelectorAll('.plotly-graph-div').forEach(function(gd) {{
            if (window.Plotly) {{ window.Plotly.Plots.resize(gd); }}
        }});
    }}

    var sel = document.getElementById('biodiscml-model-selector');
    sel.addEventListener('change', function(e) {{ selectModel(parseInt(e.target.value, 10)); }});
    selectModel(0);
}})();
</script>
"""
    display(HTML(selector_html))